# Stage 6: EmbQA-lite

Reimplements EmbQA's embedding-level reranker and entropy-based answer selection on one **train** slice (plan steps 4 to 6), and saves what the analysis needs.

- **Step 4.** The reader gives K = 2 candidates from the first retriever's top 10; keep the lowest-entropy one. References: a plain single answer (retrieval only), the first candidate, and the best candidate by F1 (upper bound of selection).
- **Step 5.** A per-question W1/W2 refiner (EmbQA Eq. 1, InfoNCE on candidate-match pseudo-labels) re-ranks the knowledge base B (top 50). Compared with the first-retriever order, a plain Contriever rerank (no refiner), and an untrained e_q + e_y (EmbQA Table 10).
- **Step 6.** Full EmbQA-lite: re-ranked top 10, then K = 2 candidates, then lowest entropy. Control: the same reader step on the plain Contriever rerank.

Not implemented: EmbQA's exploratory embedding.

**How to run (CUDA machine)**

1. Needs `data/corpus/pooled_corpus.parquet` (`01_construct_corpus`), `data/runs/<retriever>_<slice>.json` (`03_retrieval`) and `results/gold/<slice>_gold.json`.
2. Run all with `SMOKE_TEST = True` (5 questions, about a minute). Then set `SMOKE_TEST = False` and run all again. Or from the repo root, in one command:
   `EMBQA_SMOKE=0 jupyter nbconvert --to notebook --execute --inplace notebooks/06_embqa/06_embqa_lite.ipynb`
3. Outputs: `results/embqa/<retriever>_<slice>_<reader>/` (summary, predictions, per-question table, re-ranked runs: small JSON, to commit) and `data/runs/embqa_<retriever>_<slice>.json` (refined run for the 2x2). Reader outputs are cached in `data/cache/embqa/`, so a re-run only re-scores.

The stand-in reader does not use the Apple Silicon GPU (MPS): with torch 2.6, generation segfaults there. On a Mac it runs on CPU.

In [ ]:
import os, sys, json, html, time, platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from eval.evaluation import write_predictions, score_answers, score_retrieval, f1_score
from retrieval.dense import encode_texts, encode_queries, load_model as load_dense
from embqa.matching import positive_mask
from embqa.refiner import RefinerConfig, refine_query, rerank

## Configuration

Environment variables override the first four settings (for one-command runs).

In [ ]:
SLICE = os.environ.get("EMBQA_SLICE", "pilot")               # train slice: "pilot" or "tune" (never Dev here)
FIRST_RETRIEVER = os.environ.get("EMBQA_RETRIEVER", "bm25")  # B = top KB_SIZE of data/runs/<FIRST_RETRIEVER>_<SLICE>.json
READER = os.environ.get("EMBQA_READER", "standin")           # "standin" (Qwen), "main" (reader/reader.py), "fake" (no LLM)
SMOKE_TEST = os.environ.get("EMBQA_SMOKE", "1") == "1"       # True: first 5 questions, outputs kept out of git

STANDIN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
K = 2              # candidates per prompt (EmbQA, following SuRe)
TOP_N = 10         # passages shown to the reader (EmbQA: top 10)
KB_SIZE = 50       # size of the knowledge base B
E_Y_MODE = "joined"   # e_y: "joined" encodes "y1, y2" once (EmbQA's candidate sentence); "mean" averages per-candidate vectors
REFINER = RefinerConfig(shape="full", optimiser="adam", lr=1e-5, steps=20, tau=0.05, neg_per_pos=5, seed=27)

In [ ]:
TAG = f"{FIRST_RETRIEVER}_{SLICE}_{READER}"
if SMOKE_TEST:
    TAG += "_smoke"
    OUT_DIR = ROOT / "data" / "tmp" / "embqa" / TAG      # smoke-test outputs stay out of git
else:
    OUT_DIR = ROOT / "results" / "embqa" / TAG
CACHE_DIR = ROOT / "data" / "cache" / "embqa" / TAG
RUNS_DIR = ROOT / "data" / "runs"
for d in [OUT_DIR / "predictions", OUT_DIR / "runs", CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if READER == "standin":
    from embqa import standin_reader as reader
    reader.load_model(model_name=STANDIN_MODEL)
elif READER == "main":
    import importlib
    reader = importlib.import_module("reader.reader")     # Stage 4 reader
elif READER == "fake":
    from embqa import fake_reader as reader
else:
    raise ValueError(f"unknown READER {READER!r}")
READER_NAME = getattr(reader, "MODEL_NAME", READER)

dense_model, _ = load_dense()
HARDWARE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else f"CPU ({platform.machine()})"
print(f"{TAG}: reader {READER_NAME}; Contriever on {next(dense_model.parameters()).device}; {HARDWARE}")

## Data

In [ ]:
with open(ROOT / "splits" / f"{SLICE}.json") as f:
    ids = json.load(f)
if SMOKE_TEST:
    ids = ids[:5]

with open(ROOT / "results" / "gold" / f"{SLICE}_gold.json") as f:
    gold_by_id = {g["_id"]: g for g in json.load(f)}
gold = [gold_by_id[q] for q in ids]
gold_answer = {q: gold_by_id[q]["answer"] for q in ids}
gold_titles = {q: {t for t, _ in gold_by_id[q]["supporting_facts"]} for q in ids}

with open(RUNS_DIR / f"{FIRST_RETRIEVER}_{SLICE}.json") as f:
    first_run = json.load(f)
kb = {q: first_run[q][:KB_SIZE] for q in ids}          # knowledge base B per question, best first

train = (load_dataset("hotpotqa/hotpot_qa", "distractor", split="train")
         .select_columns(["id", "question", "type"]).to_pandas().set_index("id"))
questions = train.loc[ids, "question"].to_dict()
qid_to_type = train.loc[ids, "type"].to_dict()

kb_pids = sorted({p for q in ids for p in kb[q]})
corpus = pd.read_parquet(ROOT / "data" / "corpus" / "pooled_corpus.parquet", columns=["pid", "title", "text"])
corpus = corpus[corpus.pid.isin(set(kb_pids))]
para = {pid: {"title": title, "text": text} for pid, title, text in zip(corpus.pid, corpus.title, corpus.text)}
pid_to_title = {pid: p["title"] for pid, p in para.items()}
assert len(para) == len(kb_pids), "run pids missing from the corpus"
del corpus, train

print(f"{len(ids)} questions {pd.Series(qid_to_type).value_counts().to_dict()}, {len(kb_pids):,} distinct passages in B")

In [ ]:
# Contriever vectors for every passage of B (same text as 03_contriever), cached
def passage_text(pid):
    return html.unescape(para[pid]["title"]) + " " + para[pid]["text"]

EMB_PATH = CACHE_DIR / "contriever_kb.npz"
if EMB_PATH.exists():
    saved = np.load(EMB_PATH)
    assert saved["pids"].tolist() == kb_pids, "cached embeddings do not match B"
    kb_emb = saved["emb"]
else:
    t0 = time.time()
    kb_emb = encode_texts([passage_text(p) for p in kb_pids], show_progress=True)
    np.savez(EMB_PATH, pids=np.array(kb_pids), emb=kb_emb)
    print(f"encoded {len(kb_pids):,} passages in {time.time() - t0:.0f}s")
row_of = {p: i for i, p in enumerate(kb_pids)}
q_emb = dict(zip(ids, encode_queries([questions[q] for q in ids])))


def kb_matrix(q):
    return kb_emb[[row_of[p] for p in kb[q]]]

## Helpers

In [ ]:
TIMINGS = {}
ANSWER_SCORES, ANSWER_PER_Q = {}, {}


def top_paragraphs(ranked, n=TOP_N):
    return [para[p] for p in ranked[:n]]


def run_reader(name, ranked_by_q, with_plain=False):
    # Candidates (and optionally a plain answer) for every question from its ranked list; cached per question
    path = CACHE_DIR / f"reader_{name}.json"
    cache = json.load(open(path)) if path.exists() else {}
    todo = [q for q in ids if q not in cache]
    t0 = time.time()
    for i, q in enumerate(tqdm(todo, desc=name)):
        paragraphs = top_paragraphs(ranked_by_q[q])
        out = {"candidates": reader.generate_candidates(questions[q], paragraphs, k=K)}
        if with_plain:
            out["plain"] = reader.answer(questions[q], paragraphs)
        cache[q] = out
        if (i + 1) % 25 == 0 or i + 1 == len(todo):
            with open(path, "w") as f:
                json.dump(cache, f)
    if todo:
        seconds = time.time() - t0
        TIMINGS[f"reader_{name}"] = {"questions": len(todo), "seconds": round(seconds, 1),
                                     "seconds_per_question": round(seconds / len(todo), 2)}
        print(f"{name}: {len(todo)} questions in {seconds:.0f}s")
    return {q: cache[q] for q in ids}


def lowest_entropy(cands):
    return min(cands, key=lambda c: c["entropy"])["answer"]          # the first one on ties


def best_by_f1(cands, gold):
    return max(cands, key=lambda c: f1_score(c["answer"], gold)[0])["answer"]


def score_system(name, answers):
    preds = write_predictions(answers, ids, OUT_DIR / "predictions" / f"{name}.json")
    ANSWER_PER_Q[name], ANSWER_SCORES[name] = score_answers(preds, gold, qid_to_type)


def answer_table(names=None):
    rows = {}
    for n in names or ANSWER_SCORES:
        s = ANSWER_SCORES[n]
        rows[n] = {f"{part} {m}": round(s[part][m], 1)
                   for part in ["overall", "bridge", "comparison"] if part in s for m in ["em", "f1"]}
    return pd.DataFrame(rows).T

## Step 4: candidates and lowest-entropy selection on the first retriever's top 10

One prompt per system: the plain answer (retrieval only) and the K = 2 candidates (EmbQA A.1) are separate calls.

In [ ]:
step4 = run_reader("first_top10", kb, with_plain=True)

score_system("retrieval_only", {q: step4[q]["plain"] for q in ids})
score_system("first_candidate", {q: step4[q]["candidates"][0]["answer"] for q in ids})
score_system("entropy_select", {q: lowest_entropy(step4[q]["candidates"]) for q in ids})
score_system("oracle_candidate", {q: best_by_f1(step4[q]["candidates"], gold_answer[q]) for q in ids})
answer_table()

In [ ]:
# Candidate statistics: how often the reader gives K distinct candidates, and their entropies
n_cands = pd.Series({q: len(step4[q]["candidates"]) for q in ids}).value_counts().sort_index().to_dict()
identical = np.mean([len({c["answer"].lower() for c in step4[q]["candidates"]}) == 1 for q in ids])
entropies = [c["entropy"] for q in ids for c in step4[q]["candidates"] if np.isfinite(c["entropy"])]
CANDIDATE_STATS = {"candidates_per_question": {int(k): int(v) for k, v in n_cands.items()},
                   "identical_candidates_pct": round(100 * float(identical), 1),
                   "mean_entropy": round(float(np.mean(entropies)), 3)}
CANDIDATE_STATS

## Step 5: refiner rerank of B

Arms, all re-ordering the same 50 passages:
`first_retriever` (B as retrieved), `dense_rerank` (Contriever with e_q, i.e. the untrained refiner), `sum_untrained` (e_q + e_y, EmbQA Table 10), `refined` (trained W1/W2), and `pseudo_labels_first` (diagnostic: candidate-matching passages first, in retriever order).

In [ ]:
def candidate_embeddings(reader_out):
    answers = {q: [c["answer"] for c in reader_out[q]["candidates"] if c["answer"]] or [""] for q in ids}
    if E_Y_MODE == "joined":
        return dict(zip(ids, encode_queries([", ".join(answers[q]) for q in ids])))
    flat = [(q, a) for q in ids for a in answers[q]]
    vecs = encode_queries([a for _, a in flat])
    grouped = {}
    for (q, _), v in zip(flat, vecs):
        grouped.setdefault(q, []).append(v)
    return {q: np.mean(grouped[q], axis=0) for q in ids}


y_emb = candidate_embeddings(step4)
ARMS = ["first_retriever", "dense_rerank", "sum_untrained", "refined", "pseudo_labels_first"]
runs = {arm: {} for arm in ARMS}
refiner_info, pos_by_q = {}, {}

t0 = time.time()
for q in tqdm(ids, desc="refiner"):
    P = kb_matrix(q)
    pos = positive_mask(top_paragraphs(kb[q], KB_SIZE), [c["answer"] for c in step4[q]["candidates"]])
    e_new, refiner_info[q] = refine_query(q_emb[q], y_emb[q], P, pos, REFINER, qid=q)
    pos_by_q[q] = pos
    runs["first_retriever"][q] = kb[q]
    runs["dense_rerank"][q] = rerank(kb[q], P, q_emb[q])
    runs["sum_untrained"][q] = rerank(kb[q], P, q_emb[q] + y_emb[q])
    runs["refined"][q] = rerank(kb[q], P, e_new)
    runs["pseudo_labels_first"][q] = ([p for p, m in zip(kb[q], pos) if m] + [p for p, m in zip(kb[q], pos) if not m])
TIMINGS["refiner"] = {"questions": len(ids), "seconds": round(time.time() - t0, 1)}

for arm, run in runs.items():
    with open(OUT_DIR / "runs" / f"{arm}.json", "w") as f:
        json.dump(run, f)
if not SMOKE_TEST:
    with open(RUNS_DIR / f"embqa_{FIRST_RETRIEVER}_{SLICE}.json", "w") as f:
        json.dump(runs["refined"], f)

RETRIEVAL_SCORES, RETRIEVAL_PER_Q = {}, {}
for arm, run in runs.items():
    RETRIEVAL_PER_Q[arm], RETRIEVAL_SCORES[arm] = score_retrieval(run, gold, qid_to_type, pid_to_title)


def retrieval_table(part="overall", cols=("mean_gold@2", "mean_gold@5", "mean_gold@10", "both@10", "any@10")):
    return pd.DataFrame({a: {c: round(RETRIEVAL_SCORES[a][part][c], 2) for c in cols}
                         for a in ARMS if part in RETRIEVAL_SCORES[a]}).T


for part in ["overall", "bridge", "comparison"]:
    print(f"\n=== {part} ===")
    print(retrieval_table(part).to_string())

In [ ]:
# Refiner status and pseudo-label quality (are the gold paragraphs the ones labelled positive?)
info = pd.DataFrame.from_dict(refiner_info, orient="index")
trained = info[info.skipped.isna()]
REFINER_STATS = {
    "status": info.skipped.fillna("trained").value_counts().to_dict(),
    "mean_positives_when_trained": round(float(trained.n_pos.mean()), 2) if len(trained) else None,
    "mean_shift_when_trained": round(float(trained["shift"].mean()), 3) if len(trained) else None,
    "loss_first_last": [round(float(trained.loss_first.mean()), 3), round(float(trained.loss_last.mean()), 3)] if len(trained) else None,
}

labels = []
for q in ids:
    pos_titles = {pid_to_title[p] for p, m in zip(kb[q], pos_by_q[q]) if m}
    kb_titles = {pid_to_title[p] for p in kb[q]}
    labels.append({"type": qid_to_type[q], "positives": len(pos_titles),
                   "gold_in_B": len(gold_titles[q] & kb_titles),
                   "gold_positive": len(gold_titles[q] & pos_titles),
                   "positives_gold": len(pos_titles & gold_titles[q])})
labels = pd.DataFrame(labels)
by_type = labels.groupby("type")[["positives", "gold_in_B", "gold_positive"]].mean().round(2)
PSEUDO_LABELS = {
    "by_type_means": by_type.to_dict(orient="index"),
    "precision_pct": round(float(100 * labels.positives_gold.sum() / max(labels.positives.sum(), 1)), 1),
    "gold_in_B_labelled_positive_pct": round(float(100 * labels.gold_positive.sum() / max(labels.gold_in_B.sum(), 1)), 1),
}
print(REFINER_STATS)
print({k: v for k, v in PSEUDO_LABELS.items() if k != "by_type_means"})
by_type

## Step 6: full EmbQA-lite

The reader reads the refined top 10 (EmbQA-lite) and, as a control, the plain Contriever top 10.

In [ ]:
step6 = run_reader("refined_top10", runs["refined"])
step6_dense = run_reader("dense_top10", runs["dense_rerank"])

score_system("embqa_lite", {q: lowest_entropy(step6[q]["candidates"]) for q in ids})
score_system("dense_rerank_entropy", {q: lowest_entropy(step6_dense[q]["candidates"]) for q in ids})
score_system("embqa_lite_oracle_candidate", {q: best_by_f1(step6[q]["candidates"], gold_answer[q]) for q in ids})

# Reader calls per question for each system (the refiner and the reranks use no LLM call)
LLM_CALLS = {"retrieval_only": 1, "first_candidate": 1, "entropy_select": 1, "oracle_candidate": 1,
             "embqa_lite": 2, "dense_rerank_entropy": 1, "embqa_lite_oracle_candidate": 2}
answer_table().assign(llm_calls=pd.Series(LLM_CALLS))

## Refiner sweep (tune slice only)

Reuses the cached step-4 candidates, so it needs no reader call. Selects the refiner settings by mean gold@10 on the **tune** slice.

In [ ]:
SWEEP = None
if SLICE == "tune":
    grid = ([("full", lr) for lr in (1e-5, 3e-5, 1e-4)] + [("diag", lr) for lr in (1e-3, 1e-2)]
            + [("scalar", lr) for lr in (1e-2, 1e-1)])
    rows = []
    for (shape, lr) in grid:
        for steps in (10, 20, 50):
            for tau in (0.05, 0.1):
                cfg = RefinerConfig(shape=shape, optimiser="adam", lr=lr, steps=steps, tau=tau)
                run = {}
                for q in ids:
                    P = kb_matrix(q)
                    e_new, _ = refine_query(q_emb[q], y_emb[q], P, pos_by_q[q], cfg, qid=q)
                    run[q] = rerank(kb[q], P, e_new)
                _, s = score_retrieval(run, gold, qid_to_type, pid_to_title)
                rows.append({"shape": shape, "lr": lr, "steps": steps, "tau": tau,
                             **{m: round(s["overall"][m], 3) for m in ("mean_gold@5", "mean_gold@10", "both@10")},
                             "bridge mean_gold@10": round(s["bridge"]["mean_gold@10"], 3)})
    SWEEP = pd.DataFrame(rows).sort_values("mean_gold@10", ascending=False)
    SWEEP.to_json(OUT_DIR / "refiner_sweep.json", orient="records", indent=1)
    display(SWEEP.head(15))
else:
    print("skipped: runs on the tune slice only")

## Save

In [ ]:
def ranks(arm, q):
    row = RETRIEVAL_PER_Q[arm].set_index("id").loc[q]
    return [None if pd.isna(row.get(c)) else int(row.get(c)) for c in ("rank_gold_1", "rank_gold_2")]


per_question = []
for q in ids:
    per_question.append({
        "id": q, "type": qid_to_type[q], "question": questions[q], "gold": gold_answer[q],
        "gold_titles": sorted(gold_titles[q]),
        "plain": step4[q]["plain"],
        "candidates_first_top10": step4[q]["candidates"],
        "candidates_refined_top10": step6[q]["candidates"],
        "candidates_dense_top10": step6_dense[q]["candidates"],
        "predictions": {name: ANSWER_PER_Q[name].set_index("id").at[q, "pred"] for name in ANSWER_SCORES},
        "em": {name: float(ANSWER_PER_Q[name].set_index("id").at[q, "em"]) for name in ANSWER_SCORES},
        "positive_titles": [pid_to_title[p] for p, m in zip(kb[q], pos_by_q[q]) if m],
        "refiner": refiner_info[q],
        "gold_ranks": {arm: ranks(arm, q) for arm in ARMS},
    })
with open(OUT_DIR / "per_question.json", "w") as f:
    json.dump(per_question, f, indent=1)

summary = {
    "tag": TAG, "slice": SLICE, "n_questions": len(ids), "smoke_test": SMOKE_TEST,
    "first_retriever": FIRST_RETRIEVER, "reader": READER, "reader_model": READER_NAME,
    "dense_encoder": "facebook/contriever-msmarco", "hardware": HARDWARE,
    "versions": {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__},
    "config": {"K": K, "top_n": TOP_N, "kb_size": KB_SIZE, "e_y_mode": E_Y_MODE, "refiner": REFINER.to_dict(),
               "decoding": "greedy, max 32 new tokens"},
    "answers": ANSWER_SCORES, "llm_calls_per_question": LLM_CALLS, "candidates": CANDIDATE_STATS,
    "retrieval": RETRIEVAL_SCORES, "refiner": REFINER_STATS, "pseudo_labels": PSEUDO_LABELS,
    "timings": TIMINGS,
}
with open(OUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"saved to {OUT_DIR.relative_to(ROOT)}")
if SMOKE_TEST:
    print("Smoke test passed. Set SMOKE_TEST = False (or EMBQA_SMOKE=0) and run all again.")

## Open choices (plan step 7)

The paper leaves these open. Values now are first choices; the tune slice decides the ones marked "tune".

| Choice | Value now | Why | Decided on |
|---|---|---|---|
| Candidates K, passages N | 2, top 10 | EmbQA (following SuRe) | fixed |
| Size of B | 50 | plan; the run files hold 50 | tune: 20 / 50 |
| Encoder | `facebook/contriever-msmarco`, mean pooling, dot product | EmbQA reranks BM25 lists with Contriever | fixed |
| Candidate embedding e_y | "y1, y2" encoded once | EmbQA encodes the candidate sentence | tune: joined / mean |
| Shape of W1, W2 | full 768 × 768 | "linear layers" | tune: full / diag / scalar |
| Initialisation | W1 = 0, W2 = I | untrained refiner = plain Contriever rerank | fixed |
| Optimiser, lr, steps | Adam, 1e-5, 20 | in a 20-question pilot debugging check (gold answer as candidate), lr ≥ 1e-4 moved e_q by more than its own norm and pushed non-answer gold paragraphs out of the top 10 | tune: sweep above |
| Temperature τ | 0.05 | Contriever's training temperature; scores span about 0.6 to 2.0 | tune: 0.05 / 0.1 |
| Negatives | 5 per positive, resampled each step | EmbQA's 5:1 | fixed |
| "Contains a candidate" | official answer normalisation; whole-word match in title + text; yes / no / empty ignored | yes / no would match almost any passage | fixed |
| Entropy | mean next-token entropy over the candidate's tokens, greedy decoding | plan's reader contract | fixed |
| Exploratory embedding | not implemented | plan: later extra | — |